In [1]:
import pandas as pd
import numpy as np

In [4]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1435_Vivek_Vihar_Delhi_DPCC_1Day.csv")

In [5]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,195.54,336.04,6.22,20.72,26.94,33.43,3.94,0.95,2.40,...,NaN,12.24,74.14,0.34,280.01,0.00,0.00,46.01,982.55,NaN
1,2024-01-02,185.09,313.51,7.71,22.07,29.78,30.27,4.32,0.76,2.97,...,NaN,11.97,71.35,0.29,271.02,0.00,0.00,67.50,982.10,NaN
2,2024-01-03,187.98,344.21,20.97,24.17,45.14,32.82,3.89,1.64,2.04,...,NaN,11.15,83.96,0.37,302.53,0.00,0.00,29.64,982.28,NaN
3,2024-01-04,242.01,407.10,21.95,34.69,45.37,32.00,6.68,1.38,2.02,...,NaN,11.97,80.12,0.34,232.15,0.00,0.00,24.20,982.14,NaN
4,2024-01-05,212.94,329.32,15.54,48.07,38.20,37.35,9.88,1.37,4.58,...,NaN,12.49,83.71,0.35,270.94,0.00,0.00,13.42,982.04,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,191.68,231.30,24.34,37.19,39.56,37.21,5.26,1.10,5.85,...,NaN,16.21,80.90,2.14,235.49,0.81,0.81,9.29,986.69,NaN
362,2024-12-28,115.14,136.17,26.76,26.96,36.10,31.05,5.52,1.14,4.41,...,NaN,16.40,84.38,1.23,283.59,0.02,0.02,16.35,986.69,NaN
363,2024-12-29,127.01,163.50,23.59,26.80,33.43,33.07,5.16,0.67,6.03,...,NaN,16.14,79.34,2.59,278.07,0.00,0.00,78.07,986.69,NaN
364,2024-12-30,117.49,156.15,15.98,29.19,28.52,35.63,4.12,0.73,8.56,...,NaN,14.78,74.81,2.24,281.45,0.00,0.00,71.94,986.74,NaN


In [6]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [7]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [8]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [9]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         195.54        336.04        6.22        20.72   
1  2024-01-02         185.09        313.51        7.71        22.07   
2  2024-01-03         187.98        344.21       20.97        24.17   
3  2024-01-04         242.01        407.10       21.95        34.69   
4  2024-01-05         212.94        329.32       15.54        48.07   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      26.94        33.43         3.94        0.95           2.40   
1      29.78        30.27         4.32        0.76           2.97   
2      45.14        32.82         3.89        1.64           2.04   
3      45.37        32.00         6.68        1.38           2.02   
4      38.20        37.35         9.88        1.37           4.58   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             1.13             3.99    12.24   74.14      0

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [11]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.125494,0.950913,-0.738277,-1.005669,-0.311611,-1.186926,-1.046070,-0.716616,-1.291873,-0.966988,-1.254082,-2.044725,1.195393,-0.860596,1.153121,0.0,0.0,-1.514984,-0.813653
1,2024-01-02,0.994459,0.763540,-0.574138,-0.923948,-0.096788,-1.365708,-1.009345,-1.012556,-1.263724,-1.048611,-1.275378,-2.079141,0.989325,-0.961131,0.982231,0.0,0.0,-1.086490,-1.066209
2,2024-01-03,1.030697,1.018860,0.886591,-0.796827,1.065070,-1.221438,-1.050902,0.358116,-1.309651,-0.596545,-1.143975,-2.183663,1.920694,-0.800274,1.581202,0.0,0.0,-1.841389,-0.965186
3,2024-01-04,1.708194,1.541892,0.994548,-0.160012,1.082468,-1.267831,-0.781261,-0.046855,-1.310638,0.376653,-0.107701,-2.079141,1.637073,-0.860596,0.243353,0.0,0.0,-1.949859,-1.043759
4,2024-01-05,1.343677,0.895025,0.288419,0.649929,0.540116,-0.965146,-0.471996,-0.062431,-1.184216,1.079867,-0.033844,-2.012859,1.902229,-0.840489,0.980710,0.0,0.0,-2.164804,-1.099883
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.077092,0.079831,1.257832,-0.008678,0.642989,-0.973067,-0.918498,-0.482978,-1.121499,-0.062856,-0.657783,-1.538685,1.694683,2.758686,0.306843,0.0,0.0,-2.247153,1.509860
362,2024-12-28,0.117337,-0.711329,1.524420,-0.627938,0.381268,-1.321579,-0.893370,-0.420675,-1.192611,-0.502364,-0.376852,-1.514467,1.951715,0.928938,1.221173,0.0,0.0,-2.106382,1.509860
363,2024-12-29,0.266178,-0.484036,1.175211,-0.637624,0.179304,-1.207294,-0.928163,-1.152739,-1.112610,-0.759791,-0.733000,-1.547608,1.579462,-0.277489,1.116244,0.0,0.0,-0.875733,1.509860
364,2024-12-30,0.146804,-0.545163,0.336890,-0.492948,-0.192097,-1.062458,-1.028674,-1.059284,-0.987669,-0.954430,-0.897934,-1.720961,1.244879,2.959758,1.180494,0.0,0.0,-0.997960,1.537922


In [12]:
df.to_excel("vivekvihar2024.xlsx",index=False)